In [1]:
import json
from toollib.unversal import *
from toollib.data_fetcher import *

In [2]:
sys_dict = get_sys_info('ath', 'inner')
country_id = sys_dict['country_id']
country_abbr = sys_dict['country_abbr']
merchant_id = sys_dict['merchant_id']
database = sys_dict['db']

In [3]:
sys_name = 'ath'
dblink='inner'

In [4]:
country_abbr = 'th'
user,passwd = 'yutiangang', 'W0vtZiHYHrp7hKun' # 泰国

In [5]:
mysql_rule = get_dbcon(sys_name, user, passwd, dblink=dblink)

In [8]:
sample = pd.read_pickle('/home/zengjunyao/to_zjy_模型训练/df_sample_v11.pkl')

In [9]:
order_list = list(sample['app_order_id'][:2].values)
order_id_list = list(set(order_list))
logger.info(f"共计输入{len(order_list)}条数据，清理重复数据后共{len(order_id_list)}条数据")
order_id_list = list2sqlstr(order_id_list)

# af系统需要剔除外部的重复调用的数据
req_record_tbl = f"(select * from {database}.t_risk_req_record where biz_type !=4 )" if sys_name == 'af' else f"{database}.t_risk_req_record"

exc_sql = f"""
           select
                "{country_id}" as country_code,
                "{country_abbr.lower()}" as country_abbr,
                "{merchant_id}" as merchant_id,
                case when a.device_type='ios' then 1
                     when a.device_type='android' then 0 end as device_type,
                a.app_order_id as order_id,
                a.phone_number as phone,
                a.id_card_number as nid,
                a.user_id as app_user_id,
                a.app_track_id as track_id,
                a.acq_channel,
                a.apply_time,
                cast(jSON_EXTRACT(work_info, '$.workYears') as SIGNED)  as workYears,
                cast(jSON_EXTRACT(work_info, '$.salaryPayFrequency') as SIGNED)  as salaryPayFrequency,
                cast(jSON_EXTRACT(personal_info, '$.email') as char)  as email,
                b.sms_records_url,
                b.app_list_url,
                b.call_records_url,
                b.calendar_records_url,     
                b.device_info,
                c.tx_id,
                c.req_data,
                cr.term,
                cr.days_per_term
           from (select * from {database}.t_app_order where app_order_id in {order_id_list}) a
           inner  join {database}.t_app_track b on a.app_track_id = b.id
           inner  join {req_record_tbl} c on a.app_order_id = c.biz_id
           inner join {database}.t_contract cr on cr.app_order_id= a.app_order_id
           where c.tx_id is not null
"""

data_info = pd.read_sql(exc_sql, con=mysql_rule, dtype={"order_id": str})

2025-05-07 09:49:29,461 - toollib.data_fetcher - INFO - 共计输入2条数据，清理重复数据后共2条数据


In [10]:
data_info

,country_code,country_abbr,merchant_id,device_type,order_id,phone,nid,app_user_id,track_id,acq_channel,...,email,sms_records_url,app_list_url,call_records_url,calendar_records_url,device_info,tx_id,req_data,term,days_per_term
0,66,th,ATH,0,1323701931170291712,0854381432,3209600232507,1317640458664894464,2302967,VIJG,...,"""kongsakon2221@gmail.com""",https://bienslcx.ueuti.com/file/download/91a65...,https://bienslcx.ueuti.com/file/download/e823b...,https://bienslcx.ueuti.com/file/download/ad180...,,"{""SIM_state"":""1"",""abis"":""arm64-v8a,armeabi-v7a...",1323701968784809984,"{""txId"": ""1323701968784809984"", ""applyInfo"": {...",1,7
1,66,th,ATH,0,1323702219562246144,0945540414,3740300555570,1261698652018532352,2302955,APFL,...,"""newbew2013@gmail.com""",https://mocdksvc.supersafecash.com/file/downlo...,https://mocdksvc.supersafecash.com/file/downlo...,https://mocdksvc.supersafecash.com/file/downlo...,,"{""SIM_state"":""1"",""abis"":""armeabi-v7a,armeabi,""...",1323702345286508544,"{""txId"": ""1323702345286508544"", ""applyInfo"": {...",1,7
